# Evon v3.3 → GGUF (Q4_K_M, Q5_K_M) for llama.cpp on Kaggle 2× T4

Converts `gnani/gnani-evon-v3.3-30b-a3b` (BF16, ~60 GB) to 4-bit / 5-bit GGUF and pushes the files to a **private** Hugging Face repo, so a Kaggle notebook can pull them and serve them with `llama-server`.

**Before Run all**
1. Colab: key icon (Secrets) on the left → add `gnani` = a Hugging Face token with **write** access → turn on *Notebook access*.
2. Runtime type: plain **CPU** is enough (no GPU needed).
3. Keep this tab open and the laptop awake and plugged in. Each quant is uploaded as soon as it is ready, so a disconnect loses only the step in progress.

The weights are read straight from Hugging Face (`--remote`), so the 60 GB BF16 checkpoint never has to fit on Colab's disk.

In [ ]:
# 1. Runtime check
!nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
# 2. Settings
import os, shutil, subprocess, json, time, ast
from pathlib import Path

MODEL_ID = "gnani/gnani-evon-v3.3-30b-a3b"
QUANTS   = ["Q4_K_M", "Q5_K_M"]   # each is quantized, uploaded, then deleted locally
WORK     = Path("/content/work"); WORK.mkdir(parents=True, exist_ok=True)
LOG      = WORK / "status.txt"

def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def free_gb(p="/content"):
    return shutil.disk_usage(p).free / 1e9

log(f"free disk: {free_gb():.0f} GB")

In [ ]:
# 3. Hugging Face login (token comes from Colab Secrets, never printed)
!pip -q install -U huggingface_hub hf_xet tokenizers "transformers>=4.56" sentencepiece safetensors
from google.colab import userdata
from huggingface_hub import login, whoami, HfApi
os.environ["HF_TOKEN"] = userdata.get("gnani")
login(os.environ["HF_TOKEN"], add_to_git_credential=False)
me = whoami()
HF_USER = me["name"]
OUT_REPO = f"{HF_USER}/gnani-evon-v3.3-30b-a3b-GGUF"
role = me.get("auth", {}).get("accessToken", {}).get("role", "?")
log(f"logged in as {HF_USER}, token role: {role}, target repo: {OUT_REPO}")
if role == "read":
    raise SystemExit("Token is read-only: make a write (or fine-grained with repo write) token.")

In [ ]:
# 4. Build llama.cpp (only the two tools we need) + converter deps
if not Path("/content/llama.cpp").exists():
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install -e /content/llama.cpp/gguf-py
NJ = os.cpu_count()
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build --target llama-quantize llama-tokenize -j {NJ} 2>&1 | tail -3
LLAMA = Path("/content/llama.cpp")
QUANTIZE = LLAMA / "build/bin/llama-quantize"
TOKENIZE = LLAMA / "build/bin/llama-tokenize"
assert QUANTIZE.exists() and TOKENIZE.exists(), "build failed, see output above"
log("llama.cpp built: " + subprocess.run(["git", "-C", str(LLAMA), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, errors="replace").stdout.strip())

## Tokenizer check (do this before spending an hour on weights)

Evon's vocabulary (163,072) differs from Nemotron 3 Nano's, so llama.cpp may not recognise its pre-tokenizer, or may map it wrongly. We export a vocab-only GGUF and compare its token ids with the Hugging Face tokenizer on Telugu, Hindi and code-mixed text. Any mismatch → stop.

In [ ]:
# 5. Vocab-only GGUF
VOCAB_GGUF = WORK / "evon-vocab.gguf"
r = subprocess.run(["python", str(LLAMA / "convert_hf_to_gguf.py"), "--remote", MODEL_ID,
                    "--vocab-only", "--outfile", str(VOCAB_GGUF)],
                   capture_output=True, text=True, errors="replace")
print(r.stdout[-3000:]); print(r.stderr[-6000:])
if r.returncode != 0:
    if "pre-tokenizer" in (r.stdout + r.stderr):
        log("STOP: llama.cpp does not recognise Evon's pre-tokenizer (see chkhsh above). Bring this output back to Claude.")
    else:
        log("STOP: vocab-only export failed, see output above.")
    raise SystemExit(1)
log("vocab-only GGUF written")

In [ ]:
# 6. Compare token ids: llama.cpp vs Hugging Face
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer
hf_tok = Tokenizer.from_file(hf_hub_download(MODEL_ID, "tokenizer.json"))

SAMPLES = [
    "అమ్మా, ఇది మోసపు కాల్ కావచ్చు. ఫోన్ కట్ చేయండి. ఎవరికీ డబ్బు పంపకండి.",
    "మీ ఆధార్ నంబర్ మనీ లాండరింగ్ కేసులో వచ్చింది, వెంటనే వీడియో కాల్‌లో ఉండండి.",
    "मैं CBI ऑफिस से बोल रहा हूँ, आपके नाम पर एक पार्सल पकड़ा गया है।",
    "Sir aapka account freeze ho jayega, abhi OTP batao warna arrest hoga.",
    "Amma nenu bank nunchi call chestunna, mee KYC update cheyali, link pampistha.",
    "Your parcel from Mumbai customs contains 200 g of MDMA. Stay on the line.",
    "నాన్నా, ఆ OTP ఎవరికీ చెప్పకు. Bank వాళ్ళు ఎప్పుడూ OTP అడగరు.",
    "Rs. 49,999 transfer to safe account 1234-5678 by 21:40 today!!",
]

BOS_ID = 1  # bos_token_id in Evon's config.json

def llama_ids(text):
    r = subprocess.run([str(TOKENIZE), "-m", str(VOCAB_GGUF), "-p", text, "--ids"],
                       capture_output=True, text=True, errors="replace")
    lines = [l for l in r.stdout.splitlines() if l.startswith("[")]
    if not lines:
        print(r.stdout[-1000:], r.stderr[-2000:]); raise SystemExit("llama-tokenize gave no ids")
    ids = ast.literal_eval(lines[-1])
    return ids[1:] if ids and ids[0] == BOS_ID else ids  # llama.cpp may prepend BOS; HF side is encoded without it

bad = 0
for s in SAMPLES:
    a = hf_tok.encode(s, add_special_tokens=False).ids
    b = llama_ids(s)
    ok = a == b
    bad += not ok
    print(("OK  " if ok else "DIFF"), len(a), len(b), s[:50])
    if not ok:
        print("   hf   :", a[:40]); print("   llama:", b[:40])
if bad:
    log(f"STOP: {bad}/{len(SAMPLES)} tokenizer mismatches. Do not quantize; bring this output back to Claude.")
    raise SystemExit(1)
log(f"tokenizer check passed on {len(SAMPLES)} samples (Telugu, Hindi, Hinglish, English)")

## Convert weights (streamed from Hugging Face) → quantize → upload

In [ ]:
# 7. Full conversion. BF16 intermediate if disk allows (best quality), else Q8_0 (slight requantize loss).
fg = free_gb()
INTER_TYPE = "bf16" if fg >= 85 else "q8_0"
INTER = WORK / f"evon-{INTER_TYPE}.gguf"
log(f"free {fg:.0f} GB → intermediate {INTER_TYPE}; converting (expect 30–90 min)")
if not INTER.exists():
    t = time.time()
    r = subprocess.run(["python", str(LLAMA / "convert_hf_to_gguf.py"), "--remote", MODEL_ID,
                        "--outtype", INTER_TYPE, "--outfile", str(INTER) + ".part"],
                       capture_output=True, text=True, errors="replace")
    print(r.stdout[-3000:]); print(r.stderr[-6000:])
    if r.returncode != 0:
        log("STOP: conversion failed, see output above."); raise SystemExit(1)
    os.rename(str(INTER) + ".part", INTER)
    log(f"converted in {(time.time()-t)/60:.0f} min, {INTER.stat().st_size/1e9:.1f} GB, free {free_gb():.0f} GB")

In [ ]:
# 8. Private repo + model card
api = HfApi()
api.create_repo(OUT_REPO, private=True, exist_ok=True)
card = f'''---
license: apache-2.0
base_model: {MODEL_ID}
tags: [gguf, llama.cpp]
---
# Evon v3.3 30B-A3B, GGUF quants (private, for testing)

Converted from [{MODEL_ID}](https://huggingface.co/{MODEL_ID}) with llama.cpp `convert_hf_to_gguf.py --remote`,
intermediate `{INTER_TYPE}`, then `llama-quantize`. No importance matrix.
Tokenizer ids matched the Hugging Face tokenizer on Telugu, Hindi, Hinglish and English samples.
'''
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=OUT_REPO)
log(f"repo ready: https://huggingface.co/{OUT_REPO}")

In [ ]:
# 9. Quantize → upload → delete, one quant at a time
extra = ["--allow-requantize"] if INTER_TYPE == "q8_0" else []
for q in QUANTS:
    out = WORK / f"gnani-evon-v3.3-30b-a3b-{q}.gguf"
    t = time.time()
    log(f"quantizing {q} …")
    r = subprocess.run([str(QUANTIZE), *extra, str(INTER), str(out), q, str(os.cpu_count())],
                       capture_output=True, text=True, errors="replace")
    print(r.stdout[-1500:]); print(r.stderr[-1500:])
    if r.returncode != 0:
        log(f"STOP: quantize {q} failed"); raise SystemExit(1)
    log(f"{q}: {out.stat().st_size/1e9:.1f} GB in {(time.time()-t)/60:.0f} min; uploading …")
    t = time.time()
    api.upload_file(path_or_fileobj=str(out), path_in_repo=out.name, repo_id=OUT_REPO)
    log(f"{q} uploaded in {(time.time()-t)/60:.0f} min")
    out.unlink()
api.upload_file(path_or_fileobj=str(LOG), path_in_repo="status.txt", repo_id=OUT_REPO)
log("ALL DONE")